# Stage 6: Retrieval (dense + BM25, fused)

Two exact, in-memory searches over the 936 items (859 chunks + 77 image records), merged with reciprocal rank
fusion. Code: `src/handbook_qa/retrieval.py` (decision 11). Reranking is the next stage, so it is off here.

Re-running: everything here reads saved results (chunks, image index, vectors) and only embeds the questions.

The questions in `spot_check_questions.json` are a handful of hand-written sanity checks, **not an evaluation**;
a proper eval set comes later.

In [1]:
import json
from dataclasses import replace

import numpy as np
import pandas as pd

from handbook_qa.config import load_settings
from handbook_qa.retrieval import BM25, Retriever, rank_order, reciprocal_rank_fusion, tokenize

settings = load_settings()
settings = replace(settings, retrieval=replace(settings.retrieval, rerank=False))
r = settings.retrieval
print(f"candidates per list: {r.candidates} | RRF k: {r.rrf_k}")
retriever = Retriever.from_settings(settings)
print(len(retriever.items), "items")

SPOT = json.loads(open("spot_check_questions.json", encoding="utf-8").read())["questions"]
pd.set_option("display.max_colwidth", 90)

candidates per list: 50 | RRF k: 60


C:\Users\shaba\Documents\extra project\paradigm_assessment\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


936 items


## BM25 tokenizer

Lowercase words, a short stopword list, a light plural strip. No stemmer yet.

In [2]:
for text in ["How do we handle refund requests?", "What is the #ask-max channel for?", "Policies for e.g. Q3 bonuses"]:
    print(f"{text!r:50} -> {tokenize(text)}")

'How do we handle refund requests?'                -> ['handle', 'refund', 'request']
'What is the #ask-max channel for?'                -> ['ask', 'max', 'channel']
'Policies for e.g. Q3 bonuses'                     -> ['policy', 'e.g', 'q3', 'bonuse']


## One question through both searches

Cosine scores sit around 0.4-0.8 while BM25 scores are unbounded, so they can't be added. RRF uses ranks only:
each list gives an item 1 / (60 + rank), and an item both lists like rises to the top.

In [3]:
question = "If I start on July 21st, how is my commission calculated during ramp?"
dense_scores = retriever.vectors @ retriever.embed_query(question)
bm25_scores = retriever.bm25.scores(tokenize(question))
dense = rank_order(dense_scores + 2.0, r.candidates)
lexical = rank_order(bm25_scores, r.candidates)
fused = reciprocal_rank_fusion([dense, lexical], r.rrf_k)
rows = []
for n, i in enumerate(sorted(fused, key=lambda i: -fused[i])[:8], start=1):
    rows.append({"fused": n, "id": retriever.items[i].id,
                 "dense rank": dense.index(i) + 1 if i in dense else None, "cosine": round(float(dense_scores[i]), 3),
                 "bm25 rank": lexical.index(i) + 1 if i in lexical else None, "bm25": round(float(bm25_scores[i]), 1),
                 "rrf": round(fused[i], 4)})
pd.DataFrame(rows)

,fused,id,dense rank,cosine,bm25 rank,bm25,rrf
0,1,growth/sales/how-we-work#05,2,0.726,1,22.6,0.0325
1,2,img-p0617-x2727,1,0.764,6,12.0,0.0315
2,3,growth/sales/how-we-work#07,4,0.692,3,16.8,0.0315
3,4,growth/sales/new-hire-onboarding#06,6,0.675,2,19.9,0.0313
4,5,growth/sales/how-we-work#09,5,0.681,4,16.1,0.0310
5,6,growth/sales/how-we-work#04,8,0.661,5,13.1,0.0301
6,7,growth/sales/new-hire-onboarding#03,3,0.694,11,7.9,0.0300
7,8,growth/sales/how-we-work#06,10,0.659,7,10.5,0.0292


## Spot-check: dense only, BM25 only, fused

Rank of the first expected item in the top 8 (blank = not in the top 8).

In [4]:
def first_rank(ids: list[str], expected: list[str]) -> int | None:
    for n, item_id in enumerate(ids, start=1):
        if any(item_id == e or item_id.startswith(e) for e in expected):
            return n
    return None


rows = []
for case in SPOT:
    q = case["question"]
    d = rank_order(retriever.vectors @ retriever.embed_query(q) + 2.0, 8)
    b = rank_order(retriever.bm25.scores(tokenize(q)), 8)
    h = [hit.item.id for hit in retriever.search(q, k=8)]
    ids = lambda idx: [retriever.items[i].id for i in idx]
    rows.append({"question": q, "dense": first_rank(ids(d), case["expected"]),
                 "bm25": first_rank(ids(b), case["expected"]), "fused": first_rank(h, case["expected"]),
                 "fused top 3": ", ".join(h[:3])})
table = pd.DataFrame(rows)
table

,question,dense,bm25,fused,fused top 3
0,Who should I ask for new artwork or merch?,1.0,1.0,1.0,"brand/art-requests#01, company/merch-store#01, company/merch-store#02"
1,"If I start on July 21st, how is my commission calculated during ramp?",1.0,2.0,2.0,"growth/sales/how-we-work#05, img-p0617-x2727, growth/sales/how-we-work#07"
2,Which communication channels are best and worst at PostHog?,1.0,4.0,1.0,"img-p0041-x258, company/communication#06, company/communication#11"
3,How much credit do Y Combinator companies get?,1.0,1.0,1.0,"brand/startups#01, brand/startups#03, brand/startups#02"
4,How do I sign my git commits?,1.0,1.0,1.0,"engineering/security#02, engineering/posthog-com/developing-the-website#13, engineerin..."
5,How do I raise an incident?,2.0,5.0,3.0,"support/support-incident-response#01, support/support-incident-response#02, engineerin..."
6,How much RAM does the 4-core codespace have?,1.0,1.0,1.0,"img-p0355-x1727, img-p0355-x1726, img-p0356-x1730"
7,How do we handle refund requests?,1.0,1.0,1.0,"growth/sales/refunds#02, growth/sales/refunds#01, growth/sales/refunds#04"
8,What are the stages of the engineering hiring process?,1.0,1.0,1.0,"people/hiring-process/engineering-hiring#03, people/hiring-process/engineering-hiring#..."
9,What is the policy for spending company money?,1.0,1.0,1.0,"people/spending-money#01, people/spending-money#03, people/spending-money#02"


In [5]:
known = table[[bool(c["expected"]) for c in SPOT]]
summary = pd.DataFrame({col: {"found in top 8": int(known[col].notna().sum()),
                              "rank 1": int((known[col] == 1).sum()),
                              "mean rank when found": round(known[col].dropna().mean(), 2)}
                        for col in ["dense", "bm25", "fused"]}).T
print(f"{len(known)} questions with a known answer")
summary

11 questions with a known answer


,found in top 8,rank 1,mean rank when found
dense,11.0,10.0,1.09
bm25,11.0,8.0,1.73
fused,11.0,9.0,1.27
